# Analysis for the Strogatz benchmark

## Prelude

In [ ]:
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import sympy

import AnalysisUtils as au

## Loading data

These are the results of Jessamine symbolic regression.

In [ ]:
full_report = pd.read_csv("Generated/full-report.csv")

In [ ]:
full_report["sympy"] = full_report.expr_original_syms.apply(lambda e: au.parse_if_needed(e))
full_report["complexity"] = full_report.sympy.apply(lambda e: au.complexity(e))
full_report["sympy_defuzz"] = full_report.sympy.apply(lambda e: au.replace_near_integer(e.evalf()))
full_report["complexity_defuzz"] = full_report.sympy_defuzz.apply(lambda e: au.complexity(e))

In [ ]:
full_report.run_set.unique()

Make rows indexable by run set, data set, and sample number.

In [ ]:
fr2 = full_report.set_index(["run_set", "data_set", "sample_num"])

In [ ]:
srb1 = fr2.loc["SRB-2026-06-25-1715-arr8"]
srb2 = fr2.loc["SRB-2026-07-13-1130"]
cht1 = fr2.loc["CHT-2026-07-02-1730"]
cht2 = fr2.loc["CHT-2026-07-13-1130"]

In [ ]:
fr2.groupby(level=["data_set"]).size()

These are the best ones overall

In [ ]:
srb1_min_mse_ixs = srb1.groupby(level=["data_set"]).mse.idxmin()
srb2_min_mse_ixs = srb2.groupby(level=["data_set"]).mse.idxmin()
cht1_min_mse_ixs = cht1.groupby(level=["data_set"]).mse.idxmin()
cht2_min_mse_ixs = cht2.groupby(level=["data_set"]).mse.idxmin()

In [ ]:
srb1.loc[srb1_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
srb2.loc[srb1_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht1.loc[cht1_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht2.loc[cht2_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
srb1.groupby(by="data_set").agg(
    complexity_mean=("complexity", "mean"),
    complexity_defuzz_mean=("complexity_defuzz", "mean"),
    complexity_std=("complexity", "std"),
    complexity_defuzz_std=("complexity_defuzz", "std"),
    complexity_min=("complexity", "min"),
    complexity_max=("complexity", "max"),
    complexity_defuzz_min=("complexity_defuzz", "min"),
    complexity_defuzz_max=("complexity_defuzz", "max"),
).sort_values("complexity_min", ascending=True)

In [ ]:
cht1.groupby(by="data_set").agg(
    complexity_mean=("complexity", "mean"),
    complexity_defuzz_mean=("complexity_defuzz", "mean"),
    complexity_std=("complexity", "std"),
    complexity_defuzz_std=("complexity_defuzz", "std"),
    complexity_min=("complexity", "min"),
    complexity_max=("complexity", "max"),
    complexity_defuzz_min=("complexity_defuzz", "min"),
    complexity_defuzz_max=("complexity_defuzz", "max"),
).sort_values("complexity_min", ascending=True)

In [ ]:
cht2.groupby(by="data_set").agg(
    complexity_mean=("complexity", "mean"),
    complexity_defuzz_mean=("complexity_defuzz", "mean"),
    complexity_std=("complexity", "std"),
    complexity_defuzz_std=("complexity_defuzz", "std"),
    complexity_min=("complexity", "min"),
    complexity_max=("complexity", "max"),
    complexity_defuzz_min=("complexity_defuzz", "min"),
    complexity_defuzz_max=("complexity_defuzz", "max"),
).sort_values("complexity_min", ascending=True)

It looks like `srb1` is good enough, no advantage to using `srb2`.
It looks like `cht1` is good but `cht2` is even better, especially for the `bacres` data sets.

In [ ]:
srb = srb1
srb_min_mse_ixs = srb1_min_mse_ixs
cht = cht2
cht_min_mse_ixs = cht2_min_mse_ixs

## Polynomials

In [ ]:
data_sets_polynomial = [
    "d_lv1",
    "d_lv2",
    "d_vdp1",
    "d_vdp2"
    ]

In [ ]:
srb.loc[data_sets_polynomial]

The best ones are exactly correct.

In [ ]:
srb.loc[min_mse_ixs].loc[data_sets_polynomial, ["mse", "complexity_defuzz", "sympy_defuzz"]]

Only one sample is not a polynomial in $x$ and $y$.

In [ ]:
(srb.loc[data_sets_polynomial]
 .sympy_defuzz.apply(lambda e: not e.is_polynomial(sympy.abc.x, sympy.abc.y))
 .groupby(level="data_set")
 .sum())

All runs on all polynomial data sets are correct up to fuzz.

In [ ]:
au.count_by_threshold(srb.loc[data_sets_polynomial])

In [ ]:
sns.displot(data=srb.loc[data_sets_polynomial],
            x="complexity",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=[False,True],
            )

In [ ]:
sns.displot(data=srb.loc[data_sets_polynomial],
            x="complexity_defuzz",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=(False, True),
            )

In [ ]:
sns.displot(data=cht.loc[data_sets_polynomial],
            x="complexity_defuzz",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=(False, True),
            )

## Rational functions

In [ ]:
data_sets_rational = [
    "d_bacres1",
    "d_bacres2",
    "d_predprey1",
    "d_predprey2"
    ]

In [ ]:
srb.loc[min_mse_ixs].loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

Using the CHT configuration, all of these very best samples are correct.

In [ ]:
cht.loc[cht_min_mse_ixs].loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

Using the SRB configuration, the bacterial respiration ones are not quite right symbolically.
`bacres1` has some imperfection in a denominator.

In [ ]:
srb_bacres1 = srb.loc[min_mse_ixs].loc["d_bacres1", "sympy"].iloc[0]
srb_bacres1

In [ ]:
au.replace_near_integer(srb_bacres1, tolerance=5e-5)

`bacres2` is actually correct, it's just hard to simplify it down all the way.

In [ ]:
srb_bacres2 = srb.loc[min_mse_ixs].loc["d_bacres2", "sympy"].iloc[0]
srb_bacres2

In [ ]:
sympy.expand(srb_bacres2)

In [ ]:
au.replace_near_integer(sympy.expand(srb_bacres2), tolerance=5e-5)

In [ ]:
sympy.nsimplify(au.replace_near_integer(sympy.expand(srb_bacres2), tolerance=5e-5), tolerance=4e-2)

In [ ]:
sympy.simplify(sympy.nsimplify(au.replace_near_integer(sympy.expand(srb_bacres2), tolerance=5e-5), tolerance=4e-2))

The majority of solutions are not rational functions.

In [ ]:
(srb.loc[data_sets_rational]
 .sympy_defuzz
 .apply(lambda e: not e.is_rational_function())
 .groupby(level="data_set").sum())

In [ ]:
au.count_by_threshold(srb.loc[data_sets_rational], threshold=1.0e-8)

In [ ]:
bacres1_rf_ixs = srb.loc["d_bacres1"].sympy_defuzz.apply(lambda e: e.is_rational_function())

In [ ]:
srb.loc["d_bacres1"][bacres1_rf_ixs].sympy_defuzz.apply(lambda e: sympy.simplify(e.evalf()))

If we cheat, all of them have to be rational functions, and more are correct.

In [ ]:
au.count_by_threshold(cht.loc[data_sets_rational], threshold=1.0e-8)

Even with cheating, only one has the correct denominator.
Most of the rest are picking up some kind of series expansion.

In [ ]:
cht.loc["d_bacres1"].sort_values("mse").sympy_defuzz.apply(lambda e: sympy.simplify(e.evalf()))

In [ ]:
sns.displot(data=srb.loc[data_sets_rational],
            x="complexity",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=[False,True],
            )

In [ ]:
sns.displot(data=srb.loc[data_sets_rational],
            x="complexity_defuzz",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=(False, True),
            )

In [ ]:
sns.displot(data=cht.loc[data_sets_rational],
            x="complexity_defuzz",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=(False, True),
            )

## Trigonometric functions, part 1

The shear flow problems are kind of a mess so we'll handle them separately

In [ ]:
data_sets_trig = [
    "d_barmag1",
    "d_barmag2",
    "d_glider1",
    "d_glider2",
    ]

These are all correct.

In [ ]:
srb.loc[min_mse_ixs].loc[data_sets_trig, ["mse", "complexity_defuzz", "sympy_defuzz"]]

All samples are essentially correct.

In [ ]:
au.count_by_threshold(srb.loc[data_sets_trig])

In [ ]:
sns.displot(data=srb.loc[data_sets_trig],
            x="complexity",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=[False,True],
            binwidth=(1.0, 0.2)
            )

In [ ]:
sns.displot(data=srb.loc[data_sets_trig],
            x="complexity_defuzz",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=(False, True),
            binwidth=(1.0, 0.2)
            )

## Trigonometric functions, part 2

The shear flow problems are kind of a mess so we'll handle them separately

In [ ]:
data_sets_sf = [
    "d_shearflow1",
    "d_shearflow2",
    #"d_sfgrid1",
    #"d_sfgrid2",
    ]

For `shearflow2`, looks like about 12/32 are correct.
For `shearflow1`, none are correct.

In [ ]:
au.count_by_threshold(srb.loc[data_sets_sf])

In [ ]:
sns.displot(data=srb.loc[data_sets_sf],
            x="complexity",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=[False,True],

            )

In [ ]:
sns.displot(data=srb.loc[data_sets_sf],
            x="complexity_defuzz",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=(False, True),
            )

If we cheat, there are a lot more correct answers.
The CHT configuration includes a full set of trig functions, including $\cot$, which saves Jessamine from having to build up $\cos / \sin$.

In [ ]:
au.count_by_threshold(cht.loc[data_sets_sf])

In [ ]:
sns.displot(data=cht.loc[data_sets_sf],
            x="complexity",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=[False,True],

            )

In [ ]:
sns.displot(data=cht.loc[data_sets_sf],
            x="complexity_defuzz",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=(False, True),
            )